# 07 - Advanced Generators and Readouts in tProc v2

**Objective:** Explore the advanced generator and readout blocks available in the QICK firmware. This notebook covers:

1. Muxed Generators (full-speed and mixer-mux) with Muxed Readouts (PFB)
2. Interpolated Generators vs. Standard Readouts (frequency matching in tProc v2 vs. v1)
3. Dynamically Configured Readouts for frequency sweeps and electrical delay calibration
4. Fast Dynamic Readout for phase-reset applications
5. IQ offset measurement and correction
6. Performance benchmarks
7. Real-time phase tracking
8. Common pitfalls and troubleshooting
9. Best practices summary

**Prerequisites:** Complete the introductory tutorials ([`00_Getting_Started.ipynb`](./00_Getting_Started.ipynb) through [`06_Generators_And_Readouts.ipynb`](./06_Generators_And_Readouts.ipynb)).

## 1. Setup

In [ ]:
# Standard imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import logging
import time

from qick import *
from qick.asm_v2 import AveragerProgramV2, QickSweep1D, AsmV2

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

# Connect to the board (adjust path as needed)
BITSTREAM_PATH = '/path/to/your/firmware.bit'  # <--- CHANGE THIS
soc = QickSoc(BITSTREAM_PATH)
soccfg = soc

# Define channel constants (adjust based on your firmware configuration)
FSGEN_CH = 0            # Full-speed generator (single tone, max bandwidth)
INTGEN_CH = 1           # Interpolated generator (fine frequency resolution)
FSMUXGEN_CH = 3         # Full-speed mux generator (8 tones, phase coherent, up to 10 Gsps)
MIXMUXGEN_CH = 4        # Mixer-mux generator (8 tones, phase coherent, up to 7 Gsps, better SFDR)

DYNRO_CH = 0            # Dynamically configured readout (axis_dyn_readout_v1)
FASTDYNRO_CH = 1        # Fast dynamic readout (axis_readout_v3, for phase-reset applications)
MUXRO_CH = [2, 3, 4, 5] # Muxed readout channels (PFB, 4 outputs used here)
STDRO_CH = 6            # Standard readout (axis_readout_v2, PYNQ-configured)

print(f"Connected to QICK board")
print(f"Available DACs: {soccfg['dacs']}")
print(f"Available ADCs: {soccfg['adcs']}")
print(f"tProc cores: {soc.num_tprocs}")
print(f"Firmware version: {soccfg.get('fw_version', 'Unknown')}")

## 2. Generator and Readout Types Overview

Understanding the different generator and readout types is crucial for optimal performance.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                         GENERATOR TYPES                                                      ║
╠════════════════════════╦═════════════════════════════════════════════════════════════════════════════════════╣
║ Generator Type         ║ Best For                                                                            ║
╠════════════════════════╬═════════════════════════════════════════════════════════════════════════════════════╣
║ Full-Speed (FS)        ║ Single tone, maximum bandwidth (10 Gsps), simple pulses                             ║
║ Interpolated (INT)     ║ Fine frequency resolution, needs mixer_freq for upconversion, shaped pulses         ║
║ Full-Speed Mux (FSMUX) ║ Multi-tone, phase-coherent, up to 8 tones @ 10 Gsps, constant envelope              ║
║ Mixer-Mux (MIXMUX)     ║ Multi-tone, phase-coherent, lower sample rate (7 Gsps), better SFDR                 ║
╠════════════════════════╩═════════════════════════════════════════════════════════════════════════════════════╣
║                                         READOUT TYPES                                                        ║
╠════════════════════════╦═════════════════════════════════════════════════════════════════════════════════════╣
║ Readout Type           ║ Best For                                                                            ║
╠════════════════════════╬═════════════════════════════════════════════════════════════════════════════════════╣
║ Standard (STD)         ║ Fixed frequency readout, simple averaging                                           ║
║ Dynamic (DYN)          ║ Sweeping frequencies, resonator spectroscopy, delay calibration                     ║
║ Fast Dynamic (FASTDYN) ║ Phase-reset applications, fast sequencing, qubit readout                            ║
║ Muxed PFB (MUX)        ║ Multi-tone readout, 64 channels, simultaneous demodulation                          ║
╚════════════════════════╩═════════════════════════════════════════════════════════════════════════════════════╝
""")

## 3. Muxed Generators and Muxed Readouts (PFB)

Muxed generators allow you to play multiple tones simultaneously, and muxed readouts (PFB) can demodulate them efficiently.

**Key concepts:**
- The `mask` parameter in `add_pulse` selects which tones are played
- The PFB readout provides finer channel spacing than standard muxed readout (64 slices vs 8)
- Linking generators to readouts with `gen_ch` enables automatic frequency matching

In [ ]:
class MuxProgram(AveragerProgramV2):
    """
    Program demonstrating muxed generator with PFB readout.
    Plays multiple tones simultaneously on a single DAC channel.
    """
    def _initialize(self, cfg):
        ro_chs = cfg['ro_chs']
        gen_ch = cfg['gen_ch']
        
        # Declare a muxed generator with multiple tones
        # The 'ro_ch' argument links it to a readout for frequency matching
        self.declare_gen(ch=gen_ch, nqz=1, ro_ch=ro_chs[0], 
                         mux_freqs=cfg['pulse_freqs'], 
                         mux_gains=cfg['pulse_gains'], 
                         mux_phases=cfg['pulse_phases'])
        
        # Declare each PFB readout channel, linking back to the same generator
        # This automatically matches readout frequencies to generator tones
        for ch, f, ph in zip(cfg['ro_chs'], cfg['pulse_freqs'], cfg['ro_phases']):
            self.declare_readout(ch=ch, length=cfg['ro_len'], freq=f, phase=ph, gen_ch=gen_ch)

        # Create a constant pulse that plays on all 4 tones simultaneously
        # The 'mask' parameter selects which tone indices to enable
        self.add_pulse(ch=gen_ch, name="mymux", 
                       style="const", 
                       length=cfg["pulse_len"],
                       mask=[0, 1, 2, 3],
                      )
        
    def _body(self, cfg):
        # Trigger readouts to start taking data before the pulse
        self.trigger(ros=cfg['ro_chs'], pins=[0], t=cfg['trig_time'], ddr4=True)
        # Play the pulse
        self.pulse(ch=cfg['gen_ch'], name="mymux", t=0)
        
# Configuration for the program
config_mux = {
    'gen_ch': FSMUXGEN_CH,
    'ro_chs': MUXRO_CH,
    'pulse_freqs': [500, 550, 600, 650],  # MHz
    'pulse_gains': [1.0] * 4,
    'pulse_phases': [0.0] * 4,
    'ro_phases': [0.0] * 4,               # Initial downconversion phases
    'trig_time': 0.4,                    # µs, trigger before pulse
    'pulse_len': 1.0,                    # µs
    'ro_len': 1.9,                       # µs, readout length
}

print("Running muxed generator with PFB readout...")
prog = MuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config_mux)
iq_list = prog.acquire_decimated(soc, rounds=1)
t = prog.get_time_axis(ro_index=0)

# Plot results for each readout channel
fig, axes = plt.subplots(len(config_mux['ro_chs']), 1, figsize=(12, 12))
for i, ch in enumerate(config_mux['ro_chs']):
    axes[i].plot(t, iq_list[i][:, 0], label="I")
    axes[i].plot(t, iq_list[i][:, 1], label="Q")
    axes[i].plot(t, np.abs(iq_list[i].dot([1, 1j])), label="Magnitude")
    axes[i].legend()
    axes[i].set_ylabel(f"Tone {i} ({config_mux['pulse_freqs'][i]} MHz)")
    axes[i].set_xlabel("Time (µs)")
plt.suptitle("Muxed Generator with PFB Readout")
plt.tight_layout()

### 3.1 Correcting IQ Phase Offsets

The raw IQ data is rotated due to cable delays and mixer imperfections. We can correct this by measuring the average phase and applying a correction to either the upconversion (pulse) phase or the downconversion (readout) phase.

In [ ]:
# Calculate average phase offset for each channel (in degrees)
phases = [np.angle(iq.mean(axis=0).dot([1, 1j]), deg=True) for iq in iq_list]
print(f"Phase offsets: {phases}")

# Option 1: Rotate the upconversion phase
config_mux['pulse_phases'] = [-x for x in phases]
prog = MuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config_mux)
iq_list_corrected_up = prog.acquire_decimated(soc, rounds=1)

# Option 2: Rotate the downconversion phase
config_mux['pulse_phases'] = [0.0] * 4
config_mux['ro_phases'] = phases
prog = MuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config_mux)
iq_list_corrected_down = prog.acquire_decimated(soc, rounds=1)

# Plot corrected data (Option 2 as example)
fig, axes = plt.subplots(len(config_mux['ro_chs']), 1, figsize=(12, 10))
for i, ch in enumerate(config_mux['ro_chs']):
    axes[i].plot(t, iq_list_corrected_down[i][:, 0], label="I")
    axes[i].plot(t, iq_list_corrected_down[i][:, 1], label="Q")
    axes[i].plot(t, np.abs(iq_list_corrected_down[i].dot([1, 1j])), label="Magnitude")
    axes[i].legend()
    axes[i].set_ylabel(f"Tone {i} (Phase-corrected)")
    axes[i].set_xlabel("Time (µs)")
plt.suptitle("Phase-Corrected Muxed Readout")
plt.tight_layout()

### 3.2 Mixer-Mux Generator Example

The mixer-mux generator uses a digital mixer-based architecture that provides better SFDR at the cost of lower sample rate.

In [ ]:
# Using the same MuxProgram class with mixer-mux generator
config_mixmux = {
    'gen_ch': MIXMUXGEN_CH,
    'ro_chs': MUXRO_CH,
    'pulse_freqs': [400, 450, 500, 550],  # MHz (lower frequencies work better)
    'pulse_gains': [0.8] * 4,             # Slightly lower gain for better linearity
    'pulse_phases': [0.0] * 4,
    'ro_phases': [0.0] * 4,
    'trig_time': 0.4,
    'pulse_len': 1.0,
    'ro_len': 1.9,
}

print("Running mixer-mux generator...")
prog_mixmux = MuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config_mixmux)
iq_list_mixmux = prog_mixmux.acquire_decimated(soc, rounds=1)

# Plot results
fig, axes = plt.subplots(len(config_mixmux['ro_chs']), 1, figsize=(12, 10))
for i in range(len(config_mixmux['ro_chs'])):
    axes[i].plot(t, np.abs(iq_list_mixmux[i].dot([1, 1j])), label=f"Channel {i}")
    axes[i].set_ylabel(f"Magnitude (Tone {i})")
    axes[i].set_xlabel("Time (µs)")
    axes[i].legend()
plt.suptitle("Mixer-Mux Generator Performance")
plt.tight_layout()

### 3.3 Playing Different Tone Combinations

You can selectively enable tones using different masks.

In [ ]:
class MuxProgramSelective(AveragerProgramV2):
    """Program demonstrating selective tone playback using masks."""
    
    def _initialize(self, cfg):
        ro_chs = cfg['ro_chs']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=1, ro_ch=ro_chs[0], 
                         mux_freqs=cfg['pulse_freqs'], 
                         mux_gains=cfg['pulse_gains'], 
                         mux_phases=cfg['pulse_phases'])
        
        for ch, f, ph in zip(cfg['ro_chs'], cfg['pulse_freqs'], cfg['ro_phases']):
            self.declare_readout(ch=ch, length=cfg['ro_len'], freq=f, phase=ph, gen_ch=gen_ch)

        # Add multiple pulses with different masks
        self.add_pulse(ch=gen_ch, name="tones_even", 
                       style="const", length=cfg["pulse_len"],
                       mask=[0, 2])  # Even-indexed tones
        
        self.add_pulse(ch=gen_ch, name="tones_odd", 
                       style="const", length=cfg["pulse_len"],
                       mask=[1, 3])  # Odd-indexed tones
        
    def _body(self, cfg):
        self.trigger(ros=cfg['ro_chs'], pins=[0], t=cfg['trig_time'], ddr4=True)
        self.pulse(ch=cfg['gen_ch'], name="tones_even", t=0)
        self.sync_all()  # Wait for pulse to finish
        self.pulse(ch=cfg['gen_ch'], name="tones_odd", t=cfg['pulse_len'] + 0.1)

config_selective = {
    'gen_ch': FSMUXGEN_CH,
    'ro_chs': MUXRO_CH[:4],
    'pulse_freqs': [500, 550, 600, 650],
    'pulse_gains': [1.0] * 4,
    'pulse_phases': [0.0] * 4,
    'ro_phases': [0.0] * 4,
    'trig_time': 0.4,
    'pulse_len': 1.0,
    'ro_len': 2.5,
}

print("Running selective tone playback...")
prog_selective = MuxProgramSelective(soccfg, reps=1, final_delay=0.5, cfg=config_selective)
iq_list_selective = prog_selective.acquire_decimated(soc, rounds=1)

# Plot showing two pulse sequences
fig, axes = plt.subplots(2, 1, figsize=(12, 10))
for i, ch in enumerate(config_selective['ro_chs']):
    mag = np.abs(iq_list_selective[i].dot([1, 1j]))
    axes[0].plot(t, mag, label=f"Tone {i} ({config_selective['pulse_freqs'][i]} MHz)")
axes[0].set_ylabel("Magnitude")
axes[0].set_xlabel("Time (µs)")
axes[0].legend()
axes[0].set_title("Selective Tone Playback: Even tones first, then odd tones")
axes[0].grid(True)

# Plot individual tone responses
for i in range(4):
    axes[1].plot(t, np.abs(iq_list_selective[i].dot([1, 1j])), label=f"Tone {i}")
axes[1].set_ylabel("Magnitude")
axes[1].set_xlabel("Time (µs)")
axes[1].legend()
axes[1].set_title("Individual Tone Responses")
axes[1].grid(True)
plt.tight_layout()

## 4. Interpolated Generator and Standard Readout

The interpolated generator uses a digital mixer to upconvert an IF signal. 

### Frequency Matching in tProc v2 vs. tProc v1

| Aspect | tProc v1 | tProc v2 |
|--------|----------|----------|
| Frequency specification | IF (intermediate frequency) | Absolute frequency |
| Mixer frequency handling | User must add to both gen and readout | Automatic |
| Common point of confusion | Forgetting to account for mixer | None (automatic) |

In tProc v2, you specify the **absolute** frequency you want to play. The software automatically calculates the correct IF by subtracting the mixer frequency. The readout frequency is also absolute.

In [ ]:
class SimpleSweepProgram(AveragerProgramV2):
    """
    Program demonstrating interpolated generator with parameter sweeps.
    Sweeps phase and gain simultaneously across multiple steps.
    """
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        # Declare an interpolated generator with a mixer frequency
        self.declare_gen(ch=gen_ch, nqz=1, ro_ch=ro_ch, mixer_freq=cfg['mixer_freq'])
        
        # Declare a standard readout (frequency is absolute)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'], freq=cfg['freq'], gen_ch=gen_ch)

        # Add a loop for sweeping parameters
        self.add_loop("myloop", self.cfg["steps"])

        # Create a flat-top pulse with Gaussian ramps
        self.add_gauss(ch=gen_ch, name="ramp", sigma=cfg['ramp_len']/10, 
                      length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch, 
                       style="flat_top", envelope="ramp", 
                       freq=cfg['freq'], length=cfg['flat_len'],
                       phase=cfg['phase'], gain=cfg['gain'])
        
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])

# Configuration
config_sweep = {
    'steps': 5,
    'gen_ch': INTGEN_CH,
    'ro_ch': STDRO_CH,
    'freq': 2000,                          # Absolute frequency (MHz)
    'mixer_freq': 1500,                    # Mixer frequency (MHz)
    'trig_time': 0.45,                     # µs
    'ro_len': 0.3,                         # µs
    'flat_len': 0.05,                      # µs
    'ramp_len': 0.2,                       # µs (total pulse length is flat_len + ramp_len)
    'phase': QickSweep1D("myloop", -360, 720),  # Sweep phase from -360° to 720°
    'gain': QickSweep1D("myloop", 0.0, 1.0)      # Sweep gain from 0 to 1
}

print("Running interpolated generator with parameter sweeps...")
prog = SimpleSweepProgram(soccfg, reps=1, final_delay=0.5, cfg=config_sweep)
iq_list = prog.acquire_decimated(soc, rounds=1)
t = prog.get_time_axis(ro_index=0)

# Plot results
plt.figure(figsize=(12, 6))
for ii, iq in enumerate(iq_list[0]):
    plt.plot(t, iq[:, 0], label=f"I value, step {ii}")
plt.legend()
plt.ylabel("Amplitude")
plt.xlabel("Time (µs)")
plt.title("Swept Phase and Gain with Interpolated Generator")
plt.grid(True)
plt.show()

## 5. Full-Speed Generator and Dynamically Configured Readout

Dynamically configured readouts allow you to change parameters (like frequency) during a sweep. This is essential for resonator spectroscopy and electrical delay calibration.

**Key steps:**
1. Define a readout configuration using `add_readoutconfig()`
2. Send the configuration using `send_readoutconfig()` before it takes effect
3. For sweeps, send the new configuration in the `exec_before` block of the loop

In [ ]:
class DynamicFreqSweepProgram(AveragerProgramV2):
    """
    Program demonstrating dynamic readout configuration for frequency sweeps.
    Used for electrical delay calibration and resonator characterization.
    """
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'])
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        # Define the readout configuration (like a pulse for the readout)
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['freq'], gen_ch=gen_ch)
        
        # The exec_before block runs before each loop iteration
        # We use it to send the updated readout config
        loopbefore = AsmV2()
        loopbefore.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)
        self.add_loop("myloop", self.cfg["steps"], exec_before=loopbefore)

        self.add_gauss(ch=gen_ch, name="ramp", sigma=cfg['ramp_len']/10, 
                      length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch, 
                       style="flat_top", envelope="ramp", 
                       freq=cfg['freq'], length=cfg['flat_len'],
                       phase=cfg['phase'], gain=cfg['gain'])
        
    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])

# Small frequency sweep to see phase rotation
START_FREQ = 100
CAL_RANGE = 1

config_dyn = {
    'steps': 5,
    'gen_ch': FSGEN_CH,
    'ro_ch': DYNRO_CH,
    'freq': QickSweep1D("myloop", START_FREQ, START_FREQ + CAL_RANGE),
    'nqz': 1,
    'trig_time': 0.4,      # µs
    'ro_len': 0.3,         # µs
    'flat_len': 0.05,      # µs
    'ramp_len': 0.2,       # µs
    'phase': 0,
    'gain': 1.0
}

print("Running dynamic frequency sweep...")
prog = DynamicFreqSweepProgram(soccfg, reps=10, final_delay=0.5, cfg=config_dyn)
iq_list = prog.acquire(soc, rounds=1)
freqs = prog.get_pulse_param('myro', 'freq', as_array=True)

# The IQ vector rotates as frequency changes due to electrical delay
plt.figure(figsize=(8, 8))
plt.plot(iq_list[0][0][:, 0], iq_list[0][0][:, 1], '.-', markersize=10)
plt.ylabel("Q")
plt.xlabel("I")
plt.title("Phase Rotation from Small Frequency Sweep")
plt.axis('equal')
plt.grid(True)
plt.show()

### 5.1 Electrical Delay Calibration

The phase rotation we observe is due to the electrical delay between the DAC and ADC. By measuring the slope of phase vs. frequency, we can calculate this delay and correct for it.

In [ ]:
# Perform a wide-range sweep for calibration
END_FREQ = 8000  # MHz
config_dyn['freq'] = QickSweep1D("myloop", START_FREQ, END_FREQ)
config_dyn['steps'] = 1001  # Number of frequency points

print("Performing frequency sweep for delay calibration...")
prog = DynamicFreqSweepProgram(soccfg, reps=100, final_delay=1.0, cfg=config_dyn)
freqs = prog.get_pulse_param('myro', 'freq', as_array=True)

# Acquire data with progress bar
start_time = time.time()
iq_list = prog.acquire(soc, rounds=1, progress=True)
iq_complex = iq_list[0][0].dot([1, 1j])
print(f"Sweep completed in {time.time() - start_time:.2f} seconds")

# Measure the phase slope (electrical delay)
phases = np.unwrap(np.angle(iq_complex)) / (2 * np.pi)
a = np.vstack([freqs, np.ones_like(freqs)]).T
phase_delay = np.linalg.lstsq(a, phases, rcond=None)[0][0]
print(f"Measured electrical delay: {phase_delay:.6f} µs")
print(f"This corresponds to {phase_delay * 1e3:.2f} ns of cable delay")

# Apply the correction
iq_rotated = iq_complex * np.exp(-1j * freqs * 2 * np.pi * phase_delay)

# Plot the corrected IQ trajectory
plt.figure(figsize=(8, 8))
plt.plot(np.real(iq_rotated), np.imag(iq_rotated), '.')
plt.ylabel("Q")
plt.xlabel("I")
plt.title("Corrected IQ Trajectory (Flat Phase)")
plt.axis('equal')
plt.grid(True)
plt.show()

# Plot amplitude and phase of the corrected data
phases_corrected = np.unwrap(np.angle(iq_rotated)) / (2 * np.pi)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.semilogy(freqs, np.abs(iq_rotated), label="Amplitude", color='blue')
ax1.set_ylabel("Amplitude", color='blue')
ax1.set_xlabel("Frequency (MHz)")
ax1.tick_params(axis='y', labelcolor='blue')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(freqs, phases_corrected, color='red', label='Phase')
ax2.set_ylabel("Phase (cycles)", color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title("Frequency Response After Delay Calibration")
fig.tight_layout()
plt.show()

## 6. Fast Dynamic Readout (Phase-Reset Applications)

The fast dynamic readout (`axis_readout_v3`) is optimized for applications requiring rapid phase resets between measurements, such as qubit readout in quantum computing experiments. It allows changing readout parameters on every trigger without reconfiguring the entire readout chain.

In [ ]:
class FastDynamicReadoutProgram(AveragerProgramV2):
    """
    Program demonstrating fast dynamic readout for rapid parameter changes.
    Ideal for phase-reset applications and fast sequencing.
    """
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'])
        
        # Fast dynamic readout doesn't need a pre-configured frequency
        # It can change on every trigger
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        # Add multiple readout configurations for fast switching
        for i, (freq, phase) in enumerate(zip(cfg['freqs'], cfg['phases'])):
            self.add_readoutconfig(ch=ro_ch, name=f"ro_conf_{i}", 
                                  freq=freq, phase=phase, gen_ch=gen_ch)

        # Add a shaped pulse
        self.add_gauss(ch=gen_ch, name="ramp", sigma=cfg['ramp_len']/10, 
                      length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                      style="flat_top", envelope="ramp",
                      freq=cfg['freqs'][0],  # Placeholder
                      length=cfg['flat_len'],
                      phase=cfg['phases'][0],
                      gain=cfg['gain'])
        
    def _body(self, cfg):
        # Fast sequence: change readout config before each trigger
        for i in range(cfg['num_steps']):
            # Update pulse parameters for this step
            self.set_pulse_param("mypulse", "freq", cfg['freqs'][i], rollback=False)
            self.set_pulse_param("mypulse", "phase", cfg['phases'][i], rollback=False)
            
            # Send new readout configuration
            self.send_readoutconfig(ch=cfg['ro_ch'], name=f"ro_conf_{i}", t=0)
            
            # Fire pulse and measure
            self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
            self.trigger(ros=[cfg['ro_ch']], pins=[i], t=cfg['trig_time'])
            self.sync_all()  # Wait for completion before next step

# Example: Fast frequency hopping
config_fast = {
    'gen_ch': FSGEN_CH,
    'ro_ch': FASTDYNRO_CH,
    'nqz': 1,
    'freqs': [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000],  # MHz
    'phases': [0] * 10,
    'gain': 0.5,
    'trig_time': 0.4,
    'ro_len': 0.5,
    'flat_len': 0.1,
    'ramp_len': 0.2,
    'num_steps': 10,
}

print("Running fast dynamic readout test (frequency hopping)...")
prog_fast = FastDynamicReadoutProgram(soccfg, reps=100, final_delay=0.5, cfg=config_fast)
iq_list_fast = prog_fast.acquire(soc, rounds=1, progress=True)

# Plot results for each frequency
plt.figure(figsize=(12, 8))
for i, freq in enumerate(config_fast['freqs']):
    iq_data = iq_list_fast[0][i]
    magnitude = np.abs(iq_data.dot([1, 1j]))
    plt.plot(prog_fast.get_time_axis(ro_index=0), magnitude, label=f"{freq} MHz")
plt.xlabel("Time (µs)")
plt.ylabel("Magnitude")
plt.title("Fast Dynamic Readout - Frequency Hopping")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Measuring IQ Offsets

Some readout blocks (especially muxed ones) introduce a small, systematic DC offset. The `acquire()` and `acquire_decimated()` methods automatically subtract these offsets from the returned data. The raw data (without subtraction) can be accessed via `get_raw()`.

In [ ]:
class TestProgram(AveragerProgramV2):
    """Simple program to measure IQ offsets across multiple readout channels."""
    
    def _initialize(self, cfg):
        for ch in cfg['ro_chs']:
            self.declare_readout(ch=ch, length=cfg['ro_len'], freq=100)

    def _body(self, cfg):
        self.trigger(ros=cfg['ro_chs'], t=0.0)

config_offsets = {
    'ro_chs': MUXRO_CH + [STDRO_CH],
    'ro_len': 100.0,  # µs, long readout to average noise
}

print("Measuring IQ offsets...")
prog = TestProgram(soccfg, reps=1000, final_delay=0.5, cfg=config_offsets)

# Acquire data (offsets are automatically subtracted)
iq_list = prog.acquire(soc, rounds=1)

# Get raw data (to see the offsets)
iq_raw = prog.get_raw()

nch = len(prog.ro_chs)
fig, axes = plt.subplots(2 * nch, 1, figsize=(12, 3 * nch))
for i, (ch, rocfg) in enumerate(prog.ro_chs.items()):
    offset = soccfg['readouts'][ch]['iq_offset']
    bins = np.linspace(offset[0] - 0.1, offset[0] + 0.1, 101)
    nsamp = rocfg['length']
    
    # Plot I offset
    axes[2 * i].hist(iq_raw[i][:, 0, 0] / nsamp, bins=bins, alpha=0.7)
    axes[2 * i].axvline(offset[0], color='red', linestyle='--', 
                       label=f'Hardware offset: {offset[0]:.3f}')
    axes[2 * i].set_title(f"Channel {ch} - I Offset")
    axes[2 * i].legend()
    
    # Plot Q offset
    axes[2 * i + 1].hist(iq_raw[i][:, 0, 1] / nsamp, bins=bins, alpha=0.7)
    axes[2 * i + 1].axvline(offset[1], color='red', linestyle='--', 
                           label=f'Hardware offset: {offset[1]:.3f}')
    axes[2 * i + 1].set_title(f"Channel {ch} - Q Offset")
    axes[2 * i + 1].legend()

plt.tight_layout()
print("\nNote: The software automatically subtracts these hardware offsets from acquired data.")
print("You can manually access raw data via prog.get_raw() if needed.")

## 8. Performance Benchmarks

Understanding the performance characteristics of different configurations helps optimize your experiments.

In [ ]:
def benchmark_readout(readout_ch, readout_type, ro_len=1.0, reps=1000):
    """Benchmark different readout configurations."""
    class BenchProgram(AveragerProgramV2):
        def _initialize(self, cfg):
            self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'], freq=100)
        def _body(self, cfg):
            self.trigger(ros=[cfg['ro_ch']], t=0)
    
    cfg = {'ro_ch': readout_ch, 'ro_len': ro_len}
    prog = BenchProgram(soccfg, reps=reps, final_delay=0, cfg=cfg)
    
    start = time.time()
    prog.acquire(soc, rounds=1)
    elapsed = time.time() - start
    
    return elapsed, reps / elapsed  # shots per second

print("Benchmarking different readout types...")
print("-" * 60)

benchmarks = []
for ro_ch, name in [(STDRO_CH, "Standard"), (DYNRO_CH, "Dynamic"), (FASTDYNRO_CH, "Fast Dynamic")]:
    try:
        elapsed, rate = benchmark_readout(ro_ch, name)
        benchmarks.append((name, elapsed, rate))
        print(f"{name:15} Readout: {elapsed:.3f}s for 1000 shots ({rate:.0f} shots/sec)")
    except Exception as e:
        print(f"{name:15} Readout: Error - {e}")

# Benchmark muxed readout (multiple channels)
class BenchMuxProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        for ch in cfg['ro_chs']:
            self.declare_readout(ch=ch, length=cfg['ro_len'], freq=100)
    def _body(self, cfg):
        self.trigger(ros=cfg['ro_chs'], t=0)

cfg_mux = {'ro_chs': MUXRO_CH, 'ro_len': 1.0}
prog_mux = BenchMuxProgram(soccfg, reps=1000, final_delay=0, cfg=cfg_mux)
start = time.time()
prog_mux.acquire(soc, rounds=1)
elapsed_mux = time.time() - start
print(f"{'Muxed (4 ch)':15} Readout: {elapsed_mux:.3f}s for 1000 shots ({1000/elapsed_mux:.0f} shots/sec)")

print("\nPerformance Notes:")
print("• Fast Dynamic readout is optimized for rapid phase resets")
print("• Muxed readout processes multiple channels simultaneously")
print("• Longer readout lengths proportionally increase acquisition time")

## 9. Real-Time Phase Tracking Example

For qubit stability measurements, tracking phase over multiple shots is crucial.

In [ ]:
class PhaseTrackingProgram(AveragerProgramV2):
    """
    Program for tracking shot-to-shot phase stability.
    Useful for qubit stability measurements and noise characterization.
    """
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'], freq=cfg['freq'], gen_ch=gen_ch)
        
        self.add_pulse(ch=gen_ch, name="probe", style="const", 
                      length=cfg['pulse_len'], freq=cfg['freq'], 
                      phase=cfg['phase'], gain=cfg['gain'])
    
    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_time'])
        self.pulse(ch=cfg['gen_ch'], name="probe", t=0)

# Track phase over many shots
n_shots = 500
config_tracking = {
    'gen_ch': FSGEN_CH,
    'ro_ch': STDRO_CH,
    'freq': 500,
    'gain': 0.3,
    'phase': 0,
    'trig_time': 0.4,
    'ro_len': 0.5,
    'pulse_len': 0.3,
}

print(f"Tracking phase over {n_shots} shots...")
prog_track = PhaseTrackingProgram(soccfg, reps=n_shots, final_delay=0.1, cfg=config_tracking)

# Acquire data shot-by-shot
iq_shots = []
for shot in range(n_shots):
    iq = prog_track.acquire_decimated(soc, rounds=1)[0]
    iq_shots.append(iq.mean(axis=0))  # Average over time samples

iq_shots = np.array(iq_shots)
phases_shots = np.angle(iq_shots.dot([1, 1j]), deg=True)

# Plot phase stability
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(phases_shots, '.-', alpha=0.5)
ax1.set_ylabel("Phase (degrees)")
ax1.set_xlabel("Shot Number")
ax1.set_title("Shot-to-Shot Phase Stability")
ax1.axhline(np.mean(phases_shots), color='red', label=f"Mean: {np.mean(phases_shots):.1f}°")
ax1.axhline(np.mean(phases_shots) + np.std(phases_shots), color='orange', 
            linestyle='--', label=f"±1σ: {np.std(phases_shots):.2f}°")
ax1.axhline(np.mean(phases_shots) - np.std(phases_shots), color='orange', linestyle='--')
ax1.legend()
ax1.grid(True)

ax2.hist(phases_shots, bins=50, alpha=0.7)
ax2.set_xlabel("Phase (degrees)")
ax2.set_ylabel("Counts")
ax2.set_title(f"Phase Distribution (σ = {np.std(phases_shots):.3f}°)")
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Phase stability: {np.std(phases_shots):.3f} degrees RMS")
print(f"This indicates {'good' if np.std(phases_shots) < 1 else 'poor'} phase stability")

## 10. Common Pitfalls and Troubleshooting

This section covers frequently encountered issues and their solutions.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                    COMMON PITFALLS AND SOLUTIONS                                                 ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                                                  ║
║  Issue                         │ Likely Cause                    │ Solution                                      ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  No signal in readout          │ Wrong physical connection       │ Check DAC→ADC loopback cables                 ║
║                                │ Generator not enabled           │ Verify gen_ch is correct                      ║
║                                │ Wrong nqz setting               │ nqz=1 for Nyquist zone 1                      ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  Phase rotates with frequency  │ Electrical delay not calibrated │ Run calibration sweep (see Section 5)         ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  Muxed readout wrong phases    │ No frequency matching           │ Set gen_ch in declare_readout()               ║
║                                │ Tone order mismatch             │ Ensure frequencies match order                ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  Interpolated gen freq off     │ Forgot mixer_freq in tProc v2   │ Use absolute frequencies, not IFs             ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  PFB channels swapped          │ Tone order mismatch             │ Verify pulse_freqs order matches ro_chs       ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  Acquisition too slow          │ Too many reps without averaging │ Use software averaging or reduce reps         ║
║                                │ Long final_delay                │ Minimize final_delay when possible            ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  Distorted pulse shape         │ Pulse too short for ramp        │ Ensure flat_len >= ramp_len/2                 ║
║                                │ DAC saturation                  │ Reduce gain                                   ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  Memory errors                 │ Too many pulses/readouts        │ Use loops and dynamic configs                 ║
║                                │ Pulse/readout length too long   │ Limit to < 2^15 samples                       ║
║  ──────────────────────────────┼─────────────────────────────────┼───────────────────────────────────────────────║
║  Fast dynamic readout issues   │ Wrong channel type              │ Use FASTDYNRO_CH, not DYNRO_CH                ║
║                                │ Missing send_readoutconfig      │ Send config before each trigger               ║
║                                                                                                                  ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
""")

## 11. Summary and Best Practices

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                         BEST PRACTICES SUMMARY                                                   ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                                                  ║
║  ✓ Always calibrate electrical delay before precision phase measurements                                         ║
║  ✓ Use muxed generators + PFB readout for multi-tone experiments                                                 ║
║  ✓ In tProc v2, specify ABSOLUTE frequencies (not IFs)                                                           ║
║  ✓ Link generators to readouts with gen_ch parameter for automatic frequency matching                            ║
║  ✓ Use dynamic readouts for frequency sweeps to save configuration time                                          ║
║  ✓ Keep pulse lengths within BRAM constraints (< 2^15 samples @ 1ns sample rate = 32.768 µs)                     ║
║  ✓ Use software averaging (reps parameter) for noise reduction                                                   ║
║  ✓ Monitor IQ offsets and apply corrections when necessary                                                       ║
║  ✓ Test new configurations with low power and short pulses first                                                 ║
║  ✓ Use progress=True for long acquisitions to monitor status                                                     ║
║                                                                                                                  ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║                                         WHEN TO USE EACH TOOL                                                    ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                                                  ║
║  • Full-Speed Generator: Single tone, maximum bandwidth, simple pulses                                           ║
║  • Interpolated Generator: Fine frequency resolution, shaped pulses                                              ║
║  • Muxed Generators: Multiple phase-coherent tones                                                               ║
║  • Standard Readout: Fixed-frequency averaging                                                                   ║
║  • Dynamic Readout: Frequency sweeps, calibration                                                                ║
║  • Fast Dynamic Readout: Phase-reset applications, rapid sequencing                                              ║
║  • PFB Readout: Multi-tone demodulation with 64 channels                                                         ║
║                                                                                                                  ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
""")

## 12. Next Steps

You have now explored the key advanced generator and readout blocks in tProc v2.

Proceed to [`08_Hardware_Buffers.ipynb`](./08_Hardware_Buffers.ipynb) to learn about the DDR4 and MR buffers for high-speed data capture.

**Key Takeaways:**

1. **tProc v2 uses absolute frequencies** - no more manual IF calculations
2. **Link generators and readouts** with `gen_ch` for automatic frequency matching
3. **Calibrate electrical delay** for accurate phase measurements
4. **Choose the right tool** based on your application requirements
5. **Monitor hardware offsets** and apply corrections as needed

**Additional Resources:**
- To learn about basic sequencing, proceed to [`01_Basic_Sequencing.ipynb`](./01_Basic_Sequencing.ipynb)
- For parameter sweeps, see [`02_Parameter_Sweeps.ipynb`](./02_Parameter_Sweeps.ipynb)
- For real-time feedback, see [`04_Real_Time_Feedback.ipynb`](./04_Real_Time_Feedback.ipynb)
- For multi-core synchronization, see [`10_Multi_Core_Synchronization.ipynb`](./10_Multi_Core_Synchronization.ipynb)